# Tensors & Graph (epoch tensors, time-resolved tensors, graph construction)

Unisce:
- `build_subject_tensors_timefeat.ipynb` (time-resolved subject tensors)
- `graph_creation.ipynb` (graph creation + UMAP)

Data: 2026-03-05

## A) Time-resolved subject tensors (da `build_subject_tensors_timefeat.ipynb`)

# Time-resolved subject tensors (windowed features)

Questo notebook crea un tensore **per soggetto** con feature che cambiano nel tempo.

Output per soggetto:
- `X`: `(n_epochs, n_windows, 61, 40)`
- `y`: `(n_epochs,)`
- `subject_id`, `session_id`, `epoch_idx`
- info: `fs`, `win_len_s`, `step_s`, `feature_cols` (se disponibili)

Salvataggio:
`data/processed/subject_tensors/subject_tensors_time/subject_XX.pt`

In [1]:
from pathlib import Path
import os, sys

# Detect project root (works if you run from notebooks/)
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("project_root:", project_root)

# Make sure we can import your project modules
sys.path.insert(0, str(project_root))

project_root: /home/daniele_u/miralis-hypergraph-imagined-speech

In [2]:
import h5py
import torch
import numpy as np
import pandas as pd

## Params

- `win_len_s`: lunghezza finestra (secondi)
- `step_s`: passo tra finestre (secondi)

Esempio default:
- win=0.5s (128 campioni)
- step=0.25s (64 campioni)

Con 1.5s a 256 Hz → in genere escono ~5 finestre.

In [ ]:
fs = 256

win_len_s = 0.5
step_s = 0.25

win = int(round(win_len_s * fs))
step = int(round(step_s * fs))

print("fs:", fs, "win:", win, "step:", step)

fs: 256 win: 128 step: 64

## Import della tua funzione di feature

Qui devi importare la funzione che TU hai già.

Deve avere questa firma:

```python
feats, feature_cols = extract_40_features_per_channel(x_win, fs)
```

dove:
- `x_win` è `np.ndarray` shape `(61, win_samples)`
- `feats` è `np.ndarray` shape `(61, 40)`
- `feature_cols` è opzionale (lista nomi feature), può essere `None`

Se nella tua repo la funzione ha un nome diverso, modifica questa cella.

In [3]:
import numpy as np

# se comprehensive_features.py è nella root del repo:
from scripts.features.comprehensive_features import (
    extract_temporal_features,
    extract_spectral_features,
    extract_functional_features,
)

# ordine fisso delle 40 feature (stabile)
FEATURE_COLS = None

def extract_40_features_per_channel(x_win: np.ndarray, fs: int = 256):
    """
    x_win: (61, win_samples)
    return:
      feats: (61, 40)
      feature_cols: list[str] (40)
    """
    global FEATURE_COLS

    n_ch = x_win.shape[0]
    rows = []

    for ch in range(n_ch):
        sig = x_win[ch]

        d = {}
        d.update(extract_temporal_features(sig))                 # 14  [oai_citation:3‡comprehensive_features.py](sediment://file_000000003ae872439dc554545be32437)
        d.update(extract_spectral_features(sig, fs=fs))          # 20  [oai_citation:4‡comprehensive_features.py](sediment://file_000000003ae872439dc554545be32437)
        d.update(extract_functional_features(x_win, ch))         # 6   [oai_citation:5‡comprehensive_features.py](sediment://file_000000003ae872439dc554545be32437)

        rows.append(d)

    # definisci colonne una volta sola, sempre stesso ordine
    if FEATURE_COLS is None:
        # Qui uso l'ordine alfabetico delle chiavi -> stabile e riproducibile
        FEATURE_COLS = sorted(rows[0].keys())
        if len(FEATURE_COLS) != 40:
            raise ValueError(f"Expected 40 features, got {len(FEATURE_COLS)}: {FEATURE_COLS}")

    feats = np.zeros((n_ch, len(FEATURE_COLS)), dtype=np.float32)
    for ch in range(n_ch):
        feats[ch, :] = np.array([rows[ch][k] for k in FEATURE_COLS], dtype=np.float32)

    return feats, FEATURE_COLS

## Load metadata

Serve `data/interim/eeg_metadata.csv` con almeno:
- `path_h5`
- `epoch_idx`
- `subject_id`
- `session_id`
- `label_idx` (o `label_id`)

In [4]:
meta_path = project_root / "data" / "interim" / "eeg_metadata.csv"
meta = pd.read_csv(meta_path)
print("meta rows:", len(meta))
print("meta cols:", list(meta.columns))

label_col = "label_idx" if "label_idx" in meta.columns else "label_id"
print("label_col:", label_col)

# Small cleanups (same style you used elsewhere)
meta["subject_id"] = pd.to_numeric(meta["subject_id"], errors="coerce").astype("Int64")
meta = meta.dropna(subset=["subject_id", "epoch_idx"])
meta["subject_id"] = meta["subject_id"].astype(int)

meta rows: 39192
meta cols: ['subject_id', 'session_id', 'epoch_idx', 'label_name', 'label_idx', 'n_channels', 'n_samples', 'fs', 'path_h5']
label_col: label_idx

## Build + save per-subject time-resolved tensors

In [6]:
import torch
import numpy as np
import h5py
from pathlib import Path

# ============================================================
# BUILD SUBJECT TENSORS (TIME) — RESUME + SKIP EXISTING
# ============================================================

RESUME = True   # True: salta soggetti già creati (file esiste). False: ricrea e sovrascrive.
VERBOSE_SKIPS = True  # stampa warning per skip epoch fuori range / troppo corti

out_dir = project_root / "data" / "processed" / "subject_tensors" / "subject_tensors_time"
out_dir.mkdir(parents=True, exist_ok=True)
print("Saving to:", out_dir)

def parse_session_id(v):
    if isinstance(v, str):
        v = v.strip().replace("S", "")
    return int(v)

# ---- canali da locs + drop refs ----
eloc_path = project_root / "src" / "io" / "ebneuro.locs"

def read_eloc_names_first61(path):
    names = []
    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]

ch_names_61 = read_eloc_names_first61(eloc_path)

EXCLUDE = set()  # tutti 61 canali H5   # ref / overlap con Cz
keep_idx = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
keep_names = [ch_names_61[i] for i in keep_idx]

print("channels total:", len(ch_names_61), "| kept:", len(keep_names), "| excluded:", sorted(list(EXCLUDE)))

# ---- feature names ----
global_feature_cols = None

subjects = sorted(meta["subject_id"].unique().tolist())
print("num subjects in meta:", len(subjects))

total_kept = 0
total_skipped_epochs = 0
total_skipped_subjects = 0

for subject_id in subjects:
    out_path = out_dir / f"subject_{int(subject_id):02d}.pt"

    if RESUME and out_path.exists():
        total_skipped_subjects += 1
        print(f"↷ skip subject {int(subject_id):02d} (already exists)")
        continue

    df_s = meta[meta["subject_id"] == subject_id].copy()

    X_epochs = []
    y_list = []
    subj_list, sess_list, ep_list = [], [], []

    skipped_epochs = 0

    for _, r in df_s.iterrows():
        h5_path = r["path_h5"]
        epoch_idx = int(r["epoch_idx"])

        with h5py.File(h5_path, "r") as f:
            n_epochs_in_file = f["data"].shape[0]

            # skip se metadata non combacia con H5
            if epoch_idx < 0 or epoch_idx >= n_epochs_in_file:
                skipped_epochs += 1
                if VERBOSE_SKIPS:
                    print(
                        f"⚠ skip epoch: subj={subject_id} | file={Path(h5_path).name} | "
                        f"epoch_idx={epoch_idx} out of range (0..{n_epochs_in_file - 1})"
                    )
                continue

            x = f["data"][epoch_idx]  # (61, T)

        x = x.astype(np.float32)

        # drop A1/A2 già sul segnale
        x = x[keep_idx, :]   # (61, T)

        T = x.shape[1]
        starts = list(range(0, T - win + 1, step))

        if len(starts) == 0:
            skipped_epochs += 1
            if VERBOSE_SKIPS:
                print(
                    f"⚠ skip epoch: subj={subject_id} | epoch_idx={epoch_idx} | "
                    f"epoch too short: T={T}, win={win}"
                )
            continue

        feat_seq = []
        bad_epoch = False

        for s in starts:
            x_win = x[:, s:s+win]  # (59, win)

            feats_out = extract_40_features_per_channel(x_win, fs)

            # supporta sia:
            # 1) feats
            # 2) (feats, feature_cols)
            if isinstance(feats_out, tuple) and len(feats_out) == 2:
                feats, feature_cols = feats_out
                if global_feature_cols is None and feature_cols is not None:
                    global_feature_cols = list(feature_cols)
            else:
                feats = feats_out

            feats = np.asarray(feats, dtype=np.float32)

            if feats.shape != (len(keep_idx), 40):
                skipped_epochs += 1
                bad_epoch = True
                if VERBOSE_SKIPS:
                    print(
                        f"⚠ skip epoch: subj={subject_id} | epoch_idx={epoch_idx} | "
                        f"bad feature shape {feats.shape}, expected ({len(keep_idx)}, 40)"
                    )
                break

            feat_seq.append(feats)

        if bad_epoch or len(feat_seq) == 0:
            continue

        feat_seq = np.stack(feat_seq, axis=0)  # (n_windows, 59, 40)
        X_epochs.append(torch.from_numpy(feat_seq).float())

        y_list.append(int(r[label_col]))
        subj_list.append(int(r["subject_id"]))
        sess_list.append(parse_session_id(r["session_id"]))
        ep_list.append(epoch_idx)

    if len(X_epochs) == 0:
        print(f"✗ subject {subject_id}: zero valid epochs | skipped_epochs={skipped_epochs} | not saving")
        total_skipped_epochs += skipped_epochs
        continue

    X = torch.stack(X_epochs, dim=0)  # (n_epochs, n_windows, 59, 40)

    save_obj = {
        "X": X,
        "y": torch.tensor(y_list, dtype=torch.long),
        "subject_id": torch.tensor(subj_list, dtype=torch.long),
        "session_id": torch.tensor(sess_list, dtype=torch.long),
        "epoch_idx": torch.tensor(ep_list, dtype=torch.long),
        "fs": fs,
        "win_len_s": win_len_s,
        "step_s": step_s,
        "ch_names": keep_names,
        "excluded_channels": sorted(list(EXCLUDE)),
        "mode": "time_resolved_windowed_features",
    }

    if global_feature_cols is not None:
        save_obj["feature_cols"] = global_feature_cols

    torch.save(save_obj, out_path)

    kept = len(X_epochs)
    total_kept += kept
    total_skipped_epochs += skipped_epochs

    print(f"✓ saved {out_path.name} | X={tuple(X.shape)} | kept={kept} | skipped_epochs={skipped_epochs}")

print("\nDone.")
print("Skipped subjects (already existed):", total_skipped_subjects)
print("Total kept epochs:", total_kept)
print("Total skipped epochs:", total_skipped_epochs)

Saving to: /home/daniele_u/miralis-hypergraph-imagined-speech/data/processed/subject_tensors/subject_tensors_time
channels total: 61 | kept: 59 | excluded: ['A1', 'A2']
num subjects in meta: 74
↷ skip subject 00 (already exists)
↷ skip subject 01 (already exists)
↷ skip subject 02 (already exists)
↷ skip subject 03 (already exists)
↷ skip subject 04 (already exists)
↷ skip subject 05 (already exists)
↷ skip subject 06 (already exists)
↷ skip subject 07 (already exists)
↷ skip subject 08 (already exists)
↷ skip subject 09 (already exists)
↷ skip subject 10 (already exists)
↷ skip subject 11 (already exists)
↷ skip subject 12 (already exists)
↷ skip subject 13 (already exists)
↷ skip subject 14 (already exists)
↷ skip subject 15 (already exists)
↷ skip subject 16 (already exists)
↷ skip subject 17 (already exists)
↷ skip subject 18 (already exists)
↷ skip subject 19 (already exists)
↷ skip subject 20 (already exists)
↷ skip subject 21 (already exists)
↷ skip subject 22 (already exists)
↷

## One subject EEG data 

In [7]:
import time
import torch
import numpy as np
import h5py
from pathlib import Path

# ---- scegli soggetto da testare ----
TEST_SUBJECT = 0  # cambia qui

# ---- canali da locs + drop refs ----
eloc_path = project_root / "src" / "io" / "ebneuro.locs"

def read_eloc_names_first61(path):
    names = []
    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]

ch_names_61 = read_eloc_names_first61(eloc_path)

EXCLUDE = set()  # tutti 61 canali H5               # ref, overlap con Cz
keep_idx = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
keep_names = [ch_names_61[i] for i in keep_idx]

print("channels total:", len(ch_names_61), "| kept:", len(keep_names), "| excluded:", EXCLUDE)

# ---- dati soggetto ----
df_s = meta[meta["subject_id"] == TEST_SUBJECT].copy()
print("subject:", TEST_SUBJECT, "epochs:", len(df_s))

t0 = time.time()

X_epochs = []
y_list = []
subj_list, sess_list, ep_list = [], [], []

PRINT_EVERY = 25
global_feature_cols = None

for i, (_, r) in enumerate(df_s.iterrows(), start=1):
    h5_path = r["path_h5"]
    epoch_idx = int(r["epoch_idx"])

    t_epoch0 = time.time()

    with h5py.File(h5_path, "r") as f:
        x = f["data"][epoch_idx]  # (61, T)

    x = x.astype(np.float32)

    # drop A1/A2 anche dal segnale (importantissimo per feature funzionali)
    x = x[keep_idx, :]            # (61, T)

    T = x.shape[1]

    starts = list(range(0, T - win + 1, step))
    if len(starts) == 0:
        raise ValueError(f"Epoch too short: T={T}, win={win}. Check fs/window.")

    feat_seq = []
    for s in starts:
        x_win = x[:, s:s+win]  # (59, win)
        feats_out = extract_40_features_per_channel(x_win, fs)

        if isinstance(feats_out, tuple) and len(feats_out) == 2:
            feats, feature_cols = feats_out
            if global_feature_cols is None and feature_cols is not None:
                global_feature_cols = list(feature_cols)
        else:
            feats = feats_out

        feats = np.asarray(feats, dtype=np.float32)

        if feats.shape != (len(keep_idx), 40):
            raise ValueError(
                f"Bad feature shape {feats.shape} at epoch {epoch_idx}. "
                f"Expected ({len(keep_idx)},40)."
            )

        feat_seq.append(feats)

    feat_seq = np.stack(feat_seq, axis=0)  # (n_windows, 59, 40)
    X_epochs.append(torch.from_numpy(feat_seq).float())

    y_list.append(int(r[label_col]))
    subj_list.append(int(r["subject_id"]))
    sess_list.append(parse_session_id(r["session_id"]))
    ep_list.append(epoch_idx)

    t_epoch = time.time() - t_epoch0
    if (i % PRINT_EVERY) == 0 or i == 1:
        elapsed = time.time() - t0
        avg = elapsed / i
        eta = avg * (len(df_s) - i)
        print(f"[{i}/{len(df_s)}] epoch_time={t_epoch:.2f}s  avg={avg:.2f}s/epoch  ETA~{eta/60:.1f} min")

X = torch.stack(X_epochs, dim=0)  # (n_epochs, n_windows, 59, 40)

save_obj = {
    "X": X,
    "y": torch.tensor(y_list, dtype=torch.long),
    "subject_id": torch.tensor(subj_list, dtype=torch.long),
    "session_id": torch.tensor(sess_list, dtype=torch.long),
    "epoch_idx": torch.tensor(ep_list, dtype=torch.long),
    "fs": fs,
    "win_len_s": win_len_s,
    "step_s": step_s,
    "ch_names": keep_names,
    "excluded_channels": sorted(list(EXCLUDE)),
}

if global_feature_cols is not None:
    save_obj["feature_cols"] = global_feature_cols

out_path = out_dir / f"subject_{TEST_SUBJECT:02d}.pt"
torch.save(save_obj, out_path)

total = time.time() - t0
print("✓ saved:", out_path)
print("X shape:", tuple(X.shape))
print(f"total time: {total/60:.2f} min  ({total:.1f} s)")

channels total: 61 | kept: 59 | excluded: {'A2', 'A1'}
subject: 0 epochs: 550

NameError: name 'win' is not defined

# 2A, Aggregated time features

In [8]:
import time
import torch
import numpy as np
import h5py
from pathlib import Path

# ---- canali da locs + drop refs ----
eloc_path = project_root / "src" / "io" / "ebneuro.locs"

def read_eloc_names_first61(path):
    names = []
    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]

ch_names_61 = read_eloc_names_first61(eloc_path)

EXCLUDE = set()  # tutti 61 canali H5  # ref/overlap
keep_idx = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
keep_names = [ch_names_61[i] for i in keep_idx]

print("channels total:", len(ch_names_61), "| kept:", len(keep_names), "| excluded:", sorted(list(EXCLUDE)))

def parse_session_id(v):
    if isinstance(v, str):
        v = v.strip().replace("S", "")
    return int(v)

channels total: 61 | kept: 59 | excluded: ['A1', 'A2']

## All subjects tensor creation

In [9]:
import torch
import numpy as np
import h5py
from pathlib import Path

RESUME = True
VERBOSE_SKIPS = True

out_dir = project_root / "data" / "processed" / "subject_tensors" / "subject_tensors_aggregated_epoch"
out_dir.mkdir(parents=True, exist_ok=True)
print("Saving to:", out_dir)

global_feature_cols = None
subjects = sorted(meta["subject_id"].unique().tolist())
print("num subjects:", len(subjects))

total_kept = 0
total_skipped_epochs = 0
total_skipped_subjects = 0

for subject_id in subjects:
    out_path = out_dir / f"subject_{int(subject_id):02d}.pt"

    if RESUME and out_path.exists():
        total_skipped_subjects += 1
        print(f"↷ skip subject {int(subject_id):02d} (already exists)")
        continue

    df_s = meta[meta["subject_id"] == subject_id]

    X_epochs = []
    y_list = []
    subj_list, sess_list, ep_list = [], [], []

    skipped_epochs = 0

    for _, r in df_s.iterrows():
        epoch_idx = int(r["epoch_idx"])
        h5_path = r["path_h5"]

        with h5py.File(h5_path, "r") as f:
            n_epochs_in_file = f["data"].shape[0]

            if epoch_idx < 0 or epoch_idx >= n_epochs_in_file:
                skipped_epochs += 1
                if VERBOSE_SKIPS:
                    print(
                        f"⚠ skip epoch: subj={subject_id} | file={Path(h5_path).name} | "
                        f"epoch_idx={epoch_idx} out of range (0..{n_epochs_in_file - 1})"
                    )
                continue

            x = f["data"][epoch_idx]  # (61, T)

        x = x.astype(np.float32)
        x = x[keep_idx, :]  # (61, T)

        feats_out = extract_40_features_per_channel(x, fs)

        if isinstance(feats_out, tuple) and len(feats_out) == 2:
            feats, feature_cols = feats_out
            if global_feature_cols is None and feature_cols is not None:
                global_feature_cols = list(feature_cols)
        else:
            feats = feats_out

        feats = np.asarray(feats, dtype=np.float32)
        if feats.shape != (len(keep_idx), 40):
            skipped_epochs += 1
            if VERBOSE_SKIPS:
                print(
                    f"⚠ skip epoch: subj={subject_id} | epoch_idx={epoch_idx} | "
                    f"bad feature shape {feats.shape}, expected ({len(keep_idx)}, 40)"
                )
            continue

        X_epochs.append(torch.from_numpy(feats).float())  # (59, 40)

        y_list.append(int(r[label_col]))
        subj_list.append(int(r["subject_id"]))
        sess_list.append(parse_session_id(r["session_id"]))
        ep_list.append(epoch_idx)

    if len(X_epochs) == 0:
        print(f"✗ subject {subject_id}: zero valid epochs | skipped_epochs={skipped_epochs} | not saving")
        total_skipped_epochs += skipped_epochs
        continue

    X = torch.stack(X_epochs, dim=0)  # (n_epochs, 59, 40)

    save_obj = {
        "X": X,
        "y": torch.tensor(y_list, dtype=torch.long),
        "subject_id": torch.tensor(subj_list, dtype=torch.long),
        "session_id": torch.tensor(sess_list, dtype=torch.long),
        "epoch_idx": torch.tensor(ep_list, dtype=torch.long),
        "fs": fs,
        "ch_names": keep_names,
        "excluded_channels": sorted(list(EXCLUDE)),
        "mode": "aggregated_epoch_single_window",
    }

    if global_feature_cols is not None:
        save_obj["feature_cols"] = global_feature_cols

    torch.save(save_obj, out_path)

    kept = len(X_epochs)
    total_kept += kept
    total_skipped_epochs += skipped_epochs

    print(f"✓ saved {out_path.name} | X={tuple(X.shape)} | kept={kept} | skipped_epochs={skipped_epochs}")

print("\nDone.")
print("Skipped subjects (already existed):", total_skipped_subjects)
print("Total kept epochs:", total_kept)
print("Total skipped epochs:", total_skipped_epochs)

Saving to: /home/daniele_u/miralis-hypergraph-imagined-speech/data/processed/subject_tensors/subject_tensors_aggregated_epoch
num subjects: 74
↷ skip subject 00 (already exists)
↷ skip subject 01 (already exists)
↷ skip subject 02 (already exists)
↷ skip subject 03 (already exists)
↷ skip subject 04 (already exists)
↷ skip subject 05 (already exists)
↷ skip subject 06 (already exists)
↷ skip subject 07 (already exists)
↷ skip subject 08 (already exists)
↷ skip subject 09 (already exists)
↷ skip subject 10 (already exists)
↷ skip subject 11 (already exists)
↷ skip subject 12 (already exists)
↷ skip subject 13 (already exists)
↷ skip subject 14 (already exists)
↷ skip subject 15 (already exists)
↷ skip subject 16 (already exists)
↷ skip subject 17 (already exists)
↷ skip subject 18 (already exists)
↷ skip subject 19 (already exists)
↷ skip subject 20 (already exists)
↷ skip subject 21 (already exists)
↷ skip subject 22 (already exists)
↷ skip subject 23 (already exists)
↷ skip subject 24

## One subject EEG features

In [ ]:
# ---- scegli soggetto da testare ----
TEST_SUBJECT = 1  # cambia qui

out_dir = project_root / "data" / "processed" / "subject_tensors" / "subject_tensors_aggregated_epoch"
out_dir.mkdir(parents=True, exist_ok=True)
print("Saving to:", out_dir)

df_s = meta[meta["subject_id"] == TEST_SUBJECT].copy()
print("subject:", TEST_SUBJECT, "epochs:", len(df_s))

t0 = time.time()

X_epochs = []
y_list = []
subj_list, sess_list, ep_list = [], [], []

PRINT_EVERY = 25
global_feature_cols = None

for i, (_, r) in enumerate(df_s.iterrows(), start=1):
    epoch_idx = int(r["epoch_idx"])
    t_epoch0 = time.time()

    with h5py.File(r["path_h5"], "r") as f:
        x = f["data"][epoch_idx]  # (61, T)

    x = x.astype(np.float32)
    x = x[keep_idx, :]  # (61, T)

    feats_out = extract_40_features_per_channel(x, fs)

    if isinstance(feats_out, tuple) and len(feats_out) == 2:
        feats, feature_cols = feats_out
        if global_feature_cols is None and feature_cols is not None:
            global_feature_cols = list(feature_cols)
    else:
        feats = feats_out

    feats = np.asarray(feats, dtype=np.float32)
    if feats.shape != (len(keep_idx), 40):
        raise ValueError(
            f"Bad feature shape {feats.shape} at epoch {epoch_idx}. "
            f"Expected ({len(keep_idx)},40)."
        )

    X_epochs.append(torch.from_numpy(feats).float())  # (59,40)

    y_list.append(int(r[label_col]))
    subj_list.append(int(r["subject_id"]))
    sess_list.append(parse_session_id(r["session_id"]))
    ep_list.append(epoch_idx)

    t_epoch = time.time() - t_epoch0
    if (i % PRINT_EVERY) == 0 or i == 1:
        elapsed = time.time() - t0
        avg = elapsed / i
        eta = avg * (len(df_s) - i)
        print(f"[{i}/{len(df_s)}] epoch_time={t_epoch:.2f}s  avg={avg:.2f}s/epoch  ETA~{eta/60:.1f} min")

X = torch.stack(X_epochs, dim=0)  # (n_epochs, 59, 40)

save_obj = {
    "X": X,
    "y": torch.tensor(y_list, dtype=torch.long),
    "subject_id": torch.tensor(subj_list, dtype=torch.long),
    "session_id": torch.tensor(sess_list, dtype=torch.long),
    "epoch_idx": torch.tensor(ep_list, dtype=torch.long),
    "fs": fs,
    "ch_names": keep_names,
    "excluded_channels": sorted(list(EXCLUDE)),
    "mode": "aggregated_epoch_single_window",
}

if global_feature_cols is not None:
    save_obj["feature_cols"] = global_feature_cols

out_path = out_dir / f"subject_{TEST_SUBJECT:02d}.pt"
torch.save(save_obj, out_path)

total = time.time() - t0
print("✓ saved:", out_path)
print("X shape:", tuple(X.shape))
print(f"total time: {total/60:.2f} min  ({total:.1f} s)")

Saving to: /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/processed/subject_tensors/subject_tensors_aggregated_epoch
subject: 1 epochs: 550
[1/550] epoch_time=0.20s  avg=0.21s/epoch  ETA~1.9 min
[25/550] epoch_time=0.17s  avg=0.19s/epoch  ETA~1.6 min
[50/550] epoch_time=0.18s  avg=0.18s/epoch  ETA~1.5 min
[75/550] epoch_time=0.34s  avg=0.19s/epoch  ETA~1.5 min
[100/550] epoch_time=0.17s  avg=0.19s/epoch  ETA~1.4 min
[125/550] epoch_time=0.17s  avg=0.18s/epoch  ETA~1.3 min
[150/550] epoch_time=0.17s  avg=0.18s/epoch  ETA~1.2 min
[175/550] epoch_time=0.17s  avg=0.18s/epoch  ETA~1.1 min
[200/550] epoch_time=0.17s  avg=0.18s/epoch  ETA~1.0 min
[225/550] epoch_time=0.17s  avg=0.18s/epoch  ETA~1.0 min
[250/550] epoch_time=0.17s  avg=0.18s/epoch  ETA~0.9 min
[275/550] epoch_time=0.17s  avg=0.18s/epoch  ETA~0.8 min
[300/550] epoch_time=0.17s  avg=0.18s/epoch  ETA~0.7 min
[325/550] epoch_time=0.21s  avg=0.18s/epoch  ETA~0.7 min
[350/550] epoch_time=0.17s  avg=0.18s/